# Strategie de questions optimales : theorie de l'information

Ce notebook reproduit le mecanisme central du **Home Task 3 officiel IOAI 2026** ("The Analytical Language of John Wilkins") : deviner un animal cache en posant des questions oui/non a un oracle (dans le vrai Home Task, l'oracle est un LLM, ici on simplifie avec une base de connaissances explicite pour se concentrer sur la strategie).

La question centrale : **comment choisir la meilleure question a poser a chaque tour** pour deviner l'animal en un minimum de questions ? La reponse vient de la theorie de l'information : on choisit la question qui reduit le plus l'incertitude (l'entropie).

Reference officielle : https://github.com/IOAI-official/IOAI-2026 (dossier `Home Task`, tache 3, oracle = `Qwen/Qwen2.5-3B-Instruct`).

In [ ]:
import numpy as np
import math

np.random.seed(0)

## 1. Le jeu du 20 questions, formalise

On a une liste d'animaux candidats, chacun avec des attributs oui/non (vole, nage, a des plumes, etc). L'animal cache est l'un d'eux. On pose des questions du type "est-ce que l'animal vole ?" et on recoit "oui" ou "non".

Chaque reponse elimine une partie des candidats. Le but : choisir a chaque tour la question qui divise le mieux l'ensemble des candidats restants.

In [ ]:
# base de connaissances simplifiee : animal -> attributs (1 = oui, 0 = non)
animaux = {
    "aigle":     {"vole": 1, "nage": 0, "a_des_plumes": 1, "vit_en_savane": 0, "est_domestique": 0, "a_4_pattes": 0},
    "moineau":   {"vole": 1, "nage": 0, "a_des_plumes": 1, "vit_en_savane": 0, "est_domestique": 0, "a_4_pattes": 0},
    "pingouin":  {"vole": 0, "nage": 1, "a_des_plumes": 1, "vit_en_savane": 0, "est_domestique": 0, "a_4_pattes": 0},
    "canard":    {"vole": 1, "nage": 1, "a_des_plumes": 1, "vit_en_savane": 0, "est_domestique": 1, "a_4_pattes": 0},
    "lion":      {"vole": 0, "nage": 0, "a_des_plumes": 0, "vit_en_savane": 1, "est_domestique": 0, "a_4_pattes": 1},
    "girafe":    {"vole": 0, "nage": 0, "a_des_plumes": 0, "vit_en_savane": 1, "est_domestique": 0, "a_4_pattes": 1},
    "vache":     {"vole": 0, "nage": 0, "a_des_plumes": 0, "vit_en_savane": 0, "est_domestique": 1, "a_4_pattes": 1},
    "chien":     {"vole": 0, "nage": 0, "a_des_plumes": 0, "vit_en_savane": 0, "est_domestique": 1, "a_4_pattes": 1},
    "dauphin":   {"vole": 0, "nage": 1, "a_des_plumes": 0, "vit_en_savane": 0, "est_domestique": 0, "a_4_pattes": 0},
    "requin":    {"vole": 0, "nage": 1, "a_des_plumes": 0, "vit_en_savane": 0, "est_domestique": 0, "a_4_pattes": 0},
}
questions = list(next(iter(animaux.values())).keys())
print("nombre d'animaux candidats:", len(animaux))
print("questions possibles:", questions)

## 2. Entropie : mesurer l'incertitude

L'entropie d'un ensemble de candidats mesure combien de "bits d'information" il manque pour identifier l'animal avec certitude. Si tous les candidats sont egalement probables, l'entropie est `log2(nombre de candidats)`.

**Memotech** : "plus il y a de candidats plausibles, plus l'entropie est haute, plus il reste de questions a poser".

In [ ]:
def entropie(candidats):
    n = len(candidats)
    if n <= 1:
        return 0.0
    # equiprobable ici, mais la formule generale est -sum(p_i * log2(p_i))
    p = 1.0 / n
    return -n * p * math.log2(p)

print("entropie avec 10 candidats:", entropie(list(animaux.keys())))
print("entropie avec 2 candidats:", entropie(["chien", "vache"]))
print("entropie avec 1 candidat:", entropie(["chien"]))

## 3. Gain d'information : choisir la meilleure question

Pour chaque question possible, on regarde comment elle separerait les candidats actuels en deux groupes (ceux qui repondraient "oui", ceux qui repondraient "non"). Le **gain d'information** est la reduction d'entropie moyenne apportee par cette question.

La meilleure question est celle qui coupe l'ensemble le plus proche de 50/50, car c'est elle qui elimine le plus de candidats en moyenne, quelle que soit la reponse.

In [ ]:
def gain_information(candidats, question):
    oui = [a for a in candidats if animaux[a][question] == 1]
    non = [a for a in candidats if animaux[a][question] == 0]
    n = len(candidats)
    if n == 0:
        return 0.0
    entropie_avant = entropie(candidats)
    entropie_apres = (len(oui) / n) * entropie(oui) + (len(non) / n) * entropie(non)
    return entropie_avant - entropie_apres

candidats_actuels = list(animaux.keys())
print(f"{'question':20s} {'gain information':>18s}")
for q in questions:
    g = gain_information(candidats_actuels, q)
    print(f"{q:20s} {g:18.3f}")

# la question avec le plus grand gain est la meilleure a poser en premier

## 4. Strategie complete : jouer la partie avec l'algorithme glouton

A chaque tour :
1. calculer le gain d'information de toutes les questions restantes
2. poser celle avec le gain maximum
3. filtrer les candidats selon la reponse recue
4. repeter jusqu'a n'avoir plus qu'un seul candidat

C'est un algorithme **glouton** (greedy) : on prend la meilleure decision locale a chaque tour, sans forcement garantir le minimum absolu de questions sur toutes les parties possibles, mais c'est une excellente heuristique en pratique (et c'est ce qui est demande dans le Home Task officiel).

In [ ]:
def jouer_partie(animal_cache, verbose=True):
    candidats = list(animaux.keys())
    questions_restantes = list(questions)
    nb_questions_posees = 0

    while len(candidats) > 1 and questions_restantes:
        gains = {q: gain_information(candidats, q) for q in questions_restantes}
        meilleure_question = max(gains, key=gains.get)
        reponse = animaux[animal_cache][meilleure_question]
        nb_questions_posees += 1

        if verbose:
            print(f"Q{nb_questions_posees}: {meilleure_question} ? -> {'oui' if reponse else 'non'} "
                  f"(gain={gains[meilleure_question]:.3f}, {len(candidats)} candidats restants)")

        candidats = [a for a in candidats if animaux[a][meilleure_question] == reponse]
        questions_restantes.remove(meilleure_question)

    if verbose:
        print(f"-> devine: {candidats[0] if candidats else '???'} en {nb_questions_posees} questions")
    return nb_questions_posees, candidats[0] if candidats else None

# on teste sur un animal precis
jouer_partie("dauphin")

In [ ]:
# on evalue la strategie sur TOUS les animaux pour avoir une moyenne robuste
resultats = {}
for animal in animaux:
    nb_q, devine = jouer_partie(animal, verbose=False)
    resultats[animal] = nb_q
    correct = "OK" if devine == animal else "ECHEC"
    print(f"{animal:12s} devine en {nb_q} questions [{correct}]")

moyenne = sum(resultats.values()) / len(resultats)
print(f"\nmoyenne de questions necessaires: {moyenne:.2f}")
print(f"pire cas: {max(resultats.values())} questions")

## 5. Pourquoi c'est mieux qu'une strategie naive

Comparons avec une strategie "naive" qui pose les questions dans un ordre fixe, sans tenir compte du gain d'information.

In [ ]:
def jouer_partie_naive(animal_cache):
    candidats = list(animaux.keys())
    nb_questions_posees = 0
    for q in questions:   # ordre fixe, pas de calcul de gain
        if len(candidats) <= 1:
            break
        reponse = animaux[animal_cache][q]
        nb_questions_posees += 1
        candidats = [a for a in candidats if animaux[a][q] == reponse]
    return nb_questions_posees

resultats_naif = {animal: jouer_partie_naive(animal) for animal in animaux}
moyenne_naif = sum(resultats_naif.values()) / len(resultats_naif)

print(f"moyenne avec strategie gloutonne (gain d'information) : {moyenne:.2f} questions")
print(f"moyenne avec ordre fixe (naif) : {moyenne_naif:.2f} questions")

## Points a retenir pour l'epreuve

- Face a un oracle couteux a interroger (LLM, humain, capteur physique...), la strategie gloutonne par gain d'information est presque toujours la premiere chose a essayer.
- Dans le vrai Home Task 3, l'oracle est un LLM (`Qwen2.5-3B-Instruct`) et les questions doivent venir d'une liste predefinie (`questions_pool.txt`), donc le vrai defi est double : **choisir la question au meilleur gain d'information ET s'assurer que le LLM y repond de facon fiable** (un LLM peut se tromper ou etre ambigu sur une question mal formulee).
- Le meme principe de gain d'information s'applique bien au-dela des jeux de devinettes : selection de features, active learning (quelles donnees etiqueter en priorite), design d'experiences.

**Memotech** : "la meilleure question est celle qui coupe la foule des suspects en deux parts egales, pas celle qui semble la plus 'intelligente'"

## Exercice (15-20 min)

1. Ajoute 5 nouveaux animaux avec leurs attributs a la base `animaux`
2. Ajoute au moins 2 nouveaux attributs (questions) qui aident a les differencier
3. Relance l'evaluation complete et compare la moyenne de questions necessaires avant/apres l'ajout
4. Bonus : modifie `gain_information` pour gerer le cas ou une question a deja ete posee (evite de la reproposer), et le cas ou plusieurs questions ont le meme gain (chosis un critere de depart, ex: la plus courte)

In [ ]:
# ecris ta solution ici



